# Tarea 4: Análisis de Datos y Optimización

## **Jessica Yovera Ramos**

**Fecha de entrega:** [VER CANVAS]

**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa los tres problemas en este notebook
- Escribe tu código en las celdas indicadas
- Ejecuta todas las celdas para verificar que tu código funciona
- Guarda tu archivo `.ipynb` en la carpeta `tareas` de tu repositorio privado de GitHub (compartido con el docente)
- Envía el enlace a tu notebook en Canvas

**⚠️ IMPORTANTE:** GitHub registra el historial de cambios de cada archivo. Tu notebook debe ser subido a GitHub **antes del plazo**. **NO** modifiques el archivo después del plazo — los cambios tardíos serán detectados y pueden resultar en penalidad.

**Integridad académica:** Esta es una tarea individual. Puedes consultar los materiales del curso, documentación de Python, herramientas de IA y discutir conceptos con compañeros, pero todo el código debe ser tuyo.

---

In [2]:
# Importaciones estándar - ejecuta esta celda primero
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, curve_fit
from io import StringIO

---
## Problema 1: Análisis de Calidad de Agua en Ríos Andino-Amazónicos (7 puntos)

Estás analizando datos de calidad de agua de estaciones de monitoreo en tres ríos de la cuenca amazónica peruana, provenientes de la red de monitoreo de la **Autoridad Nacional del Agua (ANA)**. El conjunto de datos contiene mediciones de temperatura, oxígeno disuelto (OD), pH y conductividad eléctrica recolectadas durante varios meses.

### Tus Tareas:

**Parte A (2 puntos):** Carga y explora los datos
1. Carga los datos del string CSV provisto abajo en un DataFrame de pandas
2. Muestra información básica del conjunto de datos (forma, tipos de datos, primeras filas)
3. Verifica valores faltantes e indica cuántos hay en cada columna
4. Convierte la columna `fecha` a formato datetime usando `pd.to_datetime()`

**Parte B (3 puntos):** Análisis de datos con agrupación
1. Calcula la media, desviación estándar, mínimo y máximo de oxígeno disuelto (`od_mg_l`) agrupado por `estacion_id`
2. Determina qué estación tiene la media más baja de oxígeno disuelto
3. Crea una nueva columna llamada `estado_od` que clasifique cada medición como:
   - "Crítico" si OD < 4 mg/L
   - "Bajo" si OD está entre 4 y 6 mg/L
   - "Adecuado" si OD está entre 6 y 8 mg/L
   - "Bueno" si OD >= 8 mg/L
4. Cuenta cuántas mediciones caen en cada categoría de `estado_od` por estación

**Parte C (2 puntos):** Filtrado y resumen
1. Filtra los datos para incluir solo mediciones donde temperatura > 20°C Y pH entre 6.5 y 8.5
2. Para este subconjunto filtrado, calcula la conductividad media por mes (pista: extrae el mes de la fecha)
3. Identifica qué combinación estación-mes tuvo el mayor número de lecturas con OD "Crítico" o "Bajo"

In [4]:
# Dataset de calidad de agua - ríos andino-amazónicos del Perú
calidad_agua_csv = (
    "estacion_id,fecha,temp_c,od_mg_l,ph,conductividad_us\n"
    "RIO_UCAYALI,2024-05-15,24.3,7.2,7.1,145\n"
    "RIO_UCAYALI,2024-05-22,25.1,6.8,7.0,152\n"
    "RIO_UCAYALI,2024-06-05,24.8,6.5,6.9,158\n"
    "RIO_UCAYALI,2024-06-19,25.5,5.8,6.8,165\n"
    "RIO_UCAYALI,2024-07-03,24.2,5.2,7.0,172\n"
    "RIO_UCAYALI,2024-07-17,23.8,4.8,7.1,168\n"
    "RIO_UCAYALI,2024-08-01,24.5,4.2,7.2,175\n"
    "RIO_UCAYALI,2024-08-15,25.2,5.0,7.0,169\n"
    "RIO_TAMBOPATA,2024-05-15,21.8,8.5,7.4,98\n"
    "RIO_TAMBOPATA,2024-05-22,22.5,8.1,7.5,105\n"
    "RIO_TAMBOPATA,2024-06-05,22.9,7.8,7.3,112\n"
    "RIO_TAMBOPATA,2024-06-19,23.4,7.2,7.2,118\n"
    "RIO_TAMBOPATA,2024-07-03,22.1,6.8,7.1,125\n"
    "RIO_TAMBOPATA,2024-07-17,21.8,6.5,7.0,121\n"
    "RIO_TAMBOPATA,2024-08-01,22.5,6.9,7.1,128\n"
    "RIO_TAMBOPATA,2024-08-15,23.1,7.2,7.2,115\n"
    "RIO_MANTARO,2024-05-15,13.1,9.5,6.5,312\n"
    "RIO_MANTARO,2024-05-22,14.2,8.8,6.4,325\n"
    "RIO_MANTARO,2024-06-05,12.8,8.2,6.3,338\n"
    "RIO_MANTARO,2024-06-19,13.5,7.5,6.2,352\n"
    "RIO_MANTARO,2024-07-03,11.9,6.8,6.0,368\n"
    "RIO_MANTARO,2024-07-17,10.8,5.9,5.9,378\n"
    "RIO_MANTARO,2024-08-01,11.5,5.2,6.1,385\n"
    "RIO_MANTARO,2024-08-15,12.3,4.8,6.2,372\n"
)

# Parte A: Carga y explora los datos
from io import StringIO
# Pista: Usa pd.read_csv(StringIO(calidad_agua_csv))

# Cargar los datos del string CSV
calidad_df = pd.read_csv(StringIO(calidad_agua_csv))
print(calidad_df.head())

# Información básica
calidad_df.info()

# Verificar valores faltantes
print("Valores faltantes por columna:")
print(calidad_df.isna().sum())

# Conversión de fecha
calidad_df['fecha'] = pd.to_datetime(calidad_df['fecha'])
print(calidad_df.head())


   estacion_id       fecha  temp_c  od_mg_l   ph  conductividad_us
0  RIO_UCAYALI  2024-05-15    24.3      7.2  7.1               145
1  RIO_UCAYALI  2024-05-22    25.1      6.8  7.0               152
2  RIO_UCAYALI  2024-06-05    24.8      6.5  6.9               158
3  RIO_UCAYALI  2024-06-19    25.5      5.8  6.8               165
4  RIO_UCAYALI  2024-07-03    24.2      5.2  7.0               172
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   estacion_id       24 non-null     object 
 1   fecha             24 non-null     object 
 2   temp_c            24 non-null     float64
 3   od_mg_l           24 non-null     float64
 4   ph                24 non-null     float64
 5   conductividad_us  24 non-null     int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 1.3+ KB
Valores faltantes por columna:
estacion_id         0
fecha

In [7]:
# Parte B: Análisis de datos con agrupación

# 1. Calculo
stats_estacion = calidad_df.groupby('estacion_id')['od_mg_l'].agg(
    ['mean', 'min', 'max', 'std']
)
print(stats_estacion)

# 2. Estación tiene la media más baja de oxígeno disuelto
estacion_min_od = stats_estacion['mean'].idxmin()
print(f"La estación con la media más baja de OD es: {estacion_min_od}")

# 3. Columna estado_od
def clasificar_od(od):
    if od < 4:
        return "Crítico"
    elif od < 6:
        return "Bajo"
    elif od < 8:
        return "Adecuado"
    else:
        return "Bueno"

calidad_df['estado_od'] = calidad_df['od_mg_l'].apply(clasificar_od)
print(calidad_df[['estacion_id', 'fecha', 'od_mg_l', 'estado_od']])

# 4. Conteo de estado_od
conteo_estado = calidad_df.groupby(['estacion_id', 'estado_od']).size().unstack(fill_value=0)
print(conteo_estado)


                 mean  min  max       std
estacion_id                              
RIO_MANTARO    7.0875  4.8  9.5  1.709166
RIO_TAMBOPATA  7.3750  6.5  8.5  0.692305
RIO_UCAYALI    5.6875  4.2  7.2  1.062931
La estación con la media más baja de OD es: RIO_UCAYALI
      estacion_id      fecha  od_mg_l estado_od
0     RIO_UCAYALI 2024-05-15      7.2  Adecuado
1     RIO_UCAYALI 2024-05-22      6.8  Adecuado
2     RIO_UCAYALI 2024-06-05      6.5  Adecuado
3     RIO_UCAYALI 2024-06-19      5.8      Bajo
4     RIO_UCAYALI 2024-07-03      5.2      Bajo
5     RIO_UCAYALI 2024-07-17      4.8      Bajo
6     RIO_UCAYALI 2024-08-01      4.2      Bajo
7     RIO_UCAYALI 2024-08-15      5.0      Bajo
8   RIO_TAMBOPATA 2024-05-15      8.5     Bueno
9   RIO_TAMBOPATA 2024-05-22      8.1     Bueno
10  RIO_TAMBOPATA 2024-06-05      7.8  Adecuado
11  RIO_TAMBOPATA 2024-06-19      7.2  Adecuado
12  RIO_TAMBOPATA 2024-07-03      6.8  Adecuado
13  RIO_TAMBOPATA 2024-07-17      6.5  Adecuado
14  RIO_TAMBOP

In [ ]:
# Parte C: Filtrado y resumen

# 1. Filtrat temp > 20°C y pH
filtro = (calidad_df['temp_c'] > 20) & calidad_df['ph'].between(6.5, 8.5)
calidad_filtrada = calidad_df[filtro].copy()

# 2. Conductividad media por mes
calidad_filtrada['mes'] = calidad_filtrada['fecha'].dt.month
cond_media_mes = calidad_filtrada.groupby('mes')['conductividad_us'].mean()
print(cond_media_mes)

# 3. Combinación estación-mes
calidad_df['mes'] = calidad_df['fecha'].dt.month
criticos_bajos = calidad_df[calidad_df['estado_od'].isin(['Crítico', 'Bajo'])]
conteo_mes = criticos_bajos.groupby(['estacion_id', 'mes']).size()
print(conteo_mes.sort_values(ascending=False))
print("Combinación con más lecturas Crítico/Bajo:", conteo_mes.idxmax(), "->", conteo_mes.max())


mes
5    125.00
6    138.25
7    146.50
8    146.75
Name: conductividad_us, dtype: float64
estacion_id  mes
RIO_MANTARO  8      2
RIO_UCAYALI  8      2
             7      2
RIO_MANTARO  7      1
RIO_UCAYALI  6      1
dtype: int64
Combinación con más lecturas Crítico/Bajo: ('RIO_MANTARO', np.int32(8)) -> 2


---
## Problema 2: Comparación Estadística de Parcelas Forestales en la Amazonía (6 puntos)

Investigadores del **INIA (Instituto Nacional de Innovación Agraria) - Estación Experimental Pucallpa** midieron la biomasa arbórea (kg) en parcelas pareadas — unas sometidas a un tratamiento de aprovechamiento forestal de impacto reducido (AFIR) y otras dejadas como control. Se quiere determinar si el tratamiento afectó significativamente la biomasa individual de los árboles y si existe relación entre el diámetro y la biomasa.

### Tus Tareas:

**Parte A (2 puntos):** Comparación de grupos de tratamiento
1. Calcula estadísticas descriptivas (media, desviación estándar, mediana) de biomasa para cada grupo
2. Realiza una prueba t de dos muestras independientes para determinar si hay diferencia significativa en la biomasa media entre parcelas control y AFIR (α = 0.05)
3. Plantea tu hipótesis nula y alternativa, reporta el estadístico t y el p-valor, y escribe una conclusión

**Parte B (2 puntos):** Análisis de correlación
1. Calcula el coeficiente de correlación de Pearson entre el DAP y la biomasa para todo el conjunto de datos
2. Evalúa si esta correlación es estadísticamente significativa (α = 0.05)
3. Interpreta la fuerza y dirección de la correlación

**Parte C (2 puntos):** Ajuste de distribución
1. Ajusta una distribución normal a los datos de biomasa de las parcelas control
2. Reporta los parámetros ajustados (μ y σ)
3. Calcula la probabilidad de que un árbol seleccionado aleatoriamente de las parcelas control tenga biomasa > 150 kg
4. ¿Qué valor de biomasa representa el percentil 90 para los árboles de parcelas control?

In [11]:
# Datos de parcelas forestales - INIA Pucallpa
np.random.seed(458)  # Para reproducibilidad

# Parcelas control: bosque sin intervención
n_control = 35
dap_control = np.random.uniform(15, 50, n_control)  # DAP en cm
biomasa_control = 0.1 * dap_control**2.2 + np.random.normal(0, 15, n_control)
biomasa_control = np.maximum(biomasa_control, 10)  # Asegurar valores positivos

# Parcelas AFIR: los árboles remanentes disponen de más recursos
n_afir = 30
dap_afir = np.random.uniform(18, 55, n_afir)  # DAP en cm
biomasa_afir = 0.12 * dap_afir**2.2 + np.random.normal(5, 18, n_afir)
biomasa_afir = np.maximum(biomasa_afir, 10)

# Crear DataFrame
bosque_df = pd.DataFrame({
    'dap_cm': np.concatenate([dap_control, dap_afir]),
    'biomasa_kg': np.concatenate([biomasa_control, biomasa_afir]),
    'tratamiento': ['Control']*n_control + ['AFIR']*n_afir
})

print(bosque_df.head())
print(f"\nEspecies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)")

      dap_cm  biomasa_kg tratamiento
0  43.208835  395.956074     Control
1  49.475851  521.291644     Control
2  19.647405   54.799539     Control
3  23.651882  119.264060     Control
4  40.431164  335.576313     Control

Especies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)


In [13]:
# Parte A: Comparación de grupos de tratamiento

#1.Estadísticas descriptivas por grupo
desc = bosque_df.groupby('tratamiento')['biomasa_kg'].agg(['mean', 'std', 'median'])
print(desc)

#2. Prueba t de dos muestras independientes
control = bosque_df.loc[bosque_df['tratamiento'] == 'Control', 'biomasa_kg']
afir = bosque_df.loc[bosque_df['tratamiento'] == 'AFIR', 'biomasa_kg']
t_stat, p_valor = stats.ttest_ind(control, afir)
print(f"t = {t_stat:.4f}, p = {p_valor:.4f}")

#3. Hipotesis y conclusion
alpha = 0.05
print("H0: mu_control = mu_afir (no hay diferencia de biomasa media)")
print("H1: mu_control != mu_afir (hay diferencia de biomasa media)")
if p_valor < alpha:
  print(f"p ({p_valor:.4f}) < alpha ({alpha}): se rechaza H0, hay diferencia significativa.")
else:
  print(f"p ({p_valor:.4f}) >= alpha ({alpha}): no se rechaza H0, no hay diferencia significativa.")


                   mean         std      median
tratamiento                                    
AFIR         325.418953  213.516434  239.121336
Control      269.642640  158.210017  232.870089
t = -1.2070, p = 0.2319
H0: mu_control = mu_afir (no hay diferencia de biomasa media)
H1: mu_control != mu_afir (hay diferencia de biomasa media)
p (0.2319) >= alpha (0.05): no se rechaza H0, no hay diferencia significativa.


In [15]:
# Parte B: Análisis de correlación

#1. Correlación de Pearson DAP vs Biomasa
r, p_valor = stats.pearsonr(bosque_df['dap_cm'], bosque_df['biomasa_kg'])
print(f"r = {r:.4f}, p = {p_valor:.6f}")

#2. Significancia
alpha = 0.05
print("Correlación significativa" if p_valor < alpha else "Correlación no significativa")

#3. Interpretación
direccion = "positiva" if r > 0 else "negativa"
fuerza = "fuerte" if abs(r) >= 0.7 else ("moderada" if abs(r) >= 0.4 else "debil")
print(f"Correlacion {fuerza} y {direccion} entre DAP y biomasa.")


r = 0.9565, p = 0.000000
Correlación significativa
Correlacion fuerte y positiva entre DAP y biomasa.


In [19]:
# Parte C: Ajuste de distribución

#1 y 2. Ajustar normal
mu, sigma  = stats.norm.fit(control)
print(f"mu = {mu:.4f}, sigma = {sigma:.4f}")

#3. Probabilidad de biomasa > 150 kg
p_mayor_150 = 1 - stats.norm.cdf(150, mu, sigma)
print(f"P(biomasa > 150 kg) = {p_mayor_150:.4f}")

#4. Percentil 90
p90 = stats.norm.ppf(0.90, mu, sigma)
print(f"Percentil 90 = {p90:.4f} kg")


mu = 269.6426, sigma = 155.9335
P(biomasa > 150 kg) = 0.7785
Percentil 90 = 469.4795 kg


---
## Problema 3: Ajuste de Curva de Respuesta a la Luz (7 puntos)

La fotosíntesis depende de la intensidad de luz siguiendo una curva de saturación. La **hipérbola rectangular** se usa comúnmente para modelar esta relación:

$$A = \frac{A_{max} \cdot I}{K + I} - R_d$$

Donde:
- $A$ = tasa de fotosíntesis neta (μmol CO₂ m⁻² s⁻¹)
- $A_{max}$ = tasa máxima de fotosíntesis a saturación de luz
- $I$ = intensidad de luz (μmol fotones m⁻² s⁻¹, PAR)
- $K$ = constante de media saturación (nivel de luz en el que A = A_max/2 - R_d)
- $R_d$ = tasa de respiración en oscuridad (CO₂ liberado cuando I = 0)

Los datos provienen de mediciones de *Cecropia sp.* ("cetico"), una especie pionera característica de la Amazonía peruana, muy importante en la regeneración de bosques perturbados.

### Tus Tareas:

**Parte A (2 puntos):** Define el modelo y la función de costo
1. Escribe una función `respuesta_luz(I, Amax, K, Rd)` que implemente la ecuación anterior
2. Escribe una función de costo `respuesta_luz_mse(params, I_datos, A_datos)` que calcule el error cuadrático medio entre las tasas de fotosíntesis observadas y predichas
3. Prueba tu función `respuesta_luz` calculando A para I = 500 con Amax=25, K=200, Rd=2

**Parte B (3 puntos):** Ajusta el modelo usando optimización
1. Usa `scipy.optimize.minimize` para encontrar los parámetros óptimos (Amax, K, Rd) que minimicen el MSE
2. Usa valores iniciales: Amax=20, K=150, Rd=1
3. Reporta los parámetros ajustados y el MSE final
4. Ajusta también el modelo usando `scipy.optimize.curve_fit` y compara los resultados

**Parte C (2 puntos):** Evalúa e interpreta el modelo
1. Calcula los valores de fotosíntesis predichos usando tus parámetros ajustados
2. Calcula R² (coeficiente de determinación) para evaluar el ajuste del modelo:
   $$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
3. Calcula el **punto de compensación lumínico** (el nivel de luz donde A = 0, es decir, la fotosíntesis iguala a la respiración). Pista: despeja I cuando A = 0
4. ¿Cuál es la tasa de fotosíntesis a saturación lumínica (Amax - Rd)?

In [20]:
# Datos de curva de respuesta a la luz - Cecropia sp. (cetico)
# Mediciones en parcela de investigación, Madre de Dios

# PAR (radiación fotosintéticamente activa) en μmol fotones m⁻² s⁻¹
par_datos = np.array([0, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000, 1200, 1500, 1800])

# Tasa de fotosíntesis neta en μmol CO₂ m⁻² s⁻¹
foto_datos = np.array([-1.8, 1.2, 4.5, 7.1, 9.2, 12.5, 14.8, 17.5, 19.2, 21.1, 22.0, 22.5, 22.8, 23.0, 23.1])

print(f"Rango PAR: {par_datos.min()} a {par_datos.max()} μmol fotones m⁻² s⁻¹")
print(f"Rango fotosíntesis: {foto_datos.min()} a {foto_datos.max()} μmol CO₂ m⁻² s⁻¹")
print("Especie: Cecropia sp. (cetico) - pionera amazónica")

Rango PAR: 0 a 1800 μmol fotones m⁻² s⁻¹
Rango fotosíntesis: -1.8 a 23.1 μmol CO₂ m⁻² s⁻¹
Especie: Cecropia sp. (cetico) - pionera amazónica


In [22]:
# Parte A: Define el modelo y la función de costo
def respuesta_luz(I, Amax, K, Rd):
  return (Amax*I) / (K+I) - Rd

def respuesta_luz_mse(params, I_datos, A_datos):
  Amax, K, Rd = params
  A_pred = respuesta_luz(I_datos, Amax, K, Rd)
  return np.mean((A_datos - A_pred)**2)

print(respuesta_luz(500, Amax=25, K=200, Rd=2))

15.857142857142858


In [23]:
# Parte B: Ajusta el modelo usando optimización
x0 = [20, 150, 1]

#1-3Optimizacion con minimize
res = minimize(respuesta_luz_mse, x0, args=(par_datos, foto_datos), method='Nelder-Mead')
Amax_m, K_m, Rd_m = res.x
print(f"minimize  -> Amax={Amax_m:.4f}, K={K_m:.4f}, Rd={Rd_m:.4f}, MSE={res.fun:.4f}")

#4. Ajuste y comparacion
popt, _ = curve_fit(respuesta_luz, par_datos, foto_datos, p0=x0)
Amax_c, K_c, Rd_c = popt
mse_c = np.mean((foto_datos - respuesta_luz(par_datos, *popt)) ** 2)
print(f"curve_fit -> Amax={Amax_c:.4f}, K={K_c:.4f}, Rd={Rd_c:.4f}, MSE={mse_c:.4f}")

minimize  -> Amax=28.4460, K=134.7544, Rd=2.6272, MSE=0.2373
curve_fit -> Amax=28.4460, K=134.7545, Rd=2.6272, MSE=0.2373


In [25]:
# Parte C: Evalúa e interpreta el modelo
Amax, K, Rd = Amax_c, K_c, Rd_c

#1. Valores predichos
foto_pred = respuesta_luz(par_datos, Amax, K, Rd)

#2. R^2
ss_res = np.sum((foto_datos - foto_pred) ** 2)
ss_tot = np.sum((foto_datos - np.mean(foto_datos)) ** 2)
r2 = 1 - ss_res / ss_tot
print(f"R² = {r2:.4f}")

#3. Pinto de compensacion luminico
I_comp = (Rd * K) / (Amax - Rd)
print(f"Punto de compensación lumínico = {I_comp:.4f} μmol fotones m^-2 s^-1")

#4. Fotosintesis a saturacion luminica
print(f"Amax - Rd = {Amax - Rd:.4f} μmol CO2 m^-2 s^-1")


R² = 0.9966
Punto de compensación lumínico = 13.7117 μmol fotones m^-2 s^-1
Amax - Rd = 25.8189 μmol CO2 m^-2 s^-1


---
## Lista de verificación para entrega

Antes de entregar, verifica que:

- [ ] Todas las celdas de código se ejecutan sin errores
- [ ] Los tres problemas están completos
- [ ] Los resultados son visibles en todas las celdas
- [ ] Tu nombre está incluido al inicio
- [ ] El archivo está guardado en la carpeta `tareas` de tu repositorio privado de GitHub
- [ ] El archivo está subido a GitHub **antes del plazo**
- [ ] El enlace a tu notebook está enviado en Canvas